# 01 — Train

**Run `00_setup.ipynb` first.** Then come here, pick a config in the next cell, and run.

All checkpoints write to Drive. The config you pick determines the checkpoint subdir and W&B run name.

**Auto-resume:** if a checkpoint exists for the chosen config, this notebook resumes from the latest one. Set `FORCE_FRESH = True` to start over.

**Backgrounding:** if you have Pro+, you can close the browser after launching the train cell — it will keep running for up to 24h. With Pro you have 12h max.

## 1. Pick a config

In [ ]:
# === EDIT THIS PER RUN ===
CONFIG = 'configs/libero_v5_run0_diagnostic.yaml'
# Other options:
#   configs/libero_v5_run1_kitchen_sink.yaml   # batch + timestamp + mean_pool
#   configs/libero_v5_run2_compressor.yaml     # batch + timestamp + Perceiver compressor
#   configs/libero_v5_run2_two_stream.yaml     # batch + timestamp + two-stream

FORCE_FRESH = False  # set True to ignore existing checkpoints and start over
# =========================

import os, yaml
REPO_DIR = '/content/memory-smolVLA'
PROJECT_ROOT = '/content/drive/MyDrive/memory-smolvla'
%cd {REPO_DIR}

with open(CONFIG) as f:
    cfg = yaml.safe_load(f)
if '_base_' in cfg:
    base_path = os.path.join(os.path.dirname(CONFIG), cfg.pop('_base_'))
    with open(base_path) as f:
        base = yaml.safe_load(f)
    def merge(a, b):
        out = dict(a)
        for k, v in b.items():
            out[k] = merge(out.get(k, {}), v) if isinstance(v, dict) and isinstance(out.get(k), dict) else v
        return out
    cfg = merge(base, cfg)

ckpt_dir = cfg['trainer']['checkpoint_dir']
run_name = cfg['trainer'].get('wandb_run_name', os.path.basename(CONFIG).replace('.yaml', ''))
print(f'Config:        {CONFIG}')
print(f'Run name:      {run_name}')
print(f'Checkpoint dir: {ckpt_dir}')
print(f'Total steps:   {cfg["trainer"]["total_steps"]}')
print(f'grad_accum:    {cfg["trainer"].get("grad_accum_steps", 1)}')
print(f'window:        {cfg.get("dataset", {}).get("max_window_size", "None (full episode)")}')

## 2. Detect existing checkpoint for resume

In [ ]:
import glob, os

RESUME_FROM = None
if not FORCE_FRESH and os.path.isdir(ckpt_dir):
    pts = sorted(glob.glob(os.path.join(ckpt_dir, 'step_*.pt')))
    if pts:
        RESUME_FROM = pts[-1]
        print(f'Will resume from: {RESUME_FROM}')
    else:
        final = os.path.join(ckpt_dir, 'final.pt')
        if os.path.exists(final):
            print(f'final.pt already exists at {final}.')
            print('Set FORCE_FRESH=True if you want to retrain. Otherwise this run is done.')
        else:
            print('No previous checkpoints — fresh start.')
else:
    print('Fresh start.' if FORCE_FRESH else f'Checkpoint dir does not exist yet: {ckpt_dir}')

## 3. Launch training

Expected runtime on L4 for the v5 configs (3000 steps, grad_accum=32 → ~96K frames seen): roughly **8–12 hours**. On A100, ~3–5h.

If the cell streams output and then disconnects, the process keeps running. Re-run cell 2 above to see latest checkpoint, then this cell with the new RESUME_FROM.

In [ ]:
import os
os.environ['MUJOCO_GL'] = 'osmesa'  # in case kernel restarted

resume_arg = f'--resume {RESUME_FROM}' if RESUME_FROM else ''

!python scripts/train.py --config {CONFIG} {resume_arg} 2>&1 | tee -a {PROJECT_ROOT}/wandb_state/{run_name}.log

## 4. Confirm checkpoint saved

In [ ]:
import os, glob
files = sorted(glob.glob(os.path.join(ckpt_dir, '*.pt')))
for f in files:
    size_mb = os.path.getsize(f) / 1e6
    print(f'{f}  ({size_mb:.0f} MB)')
print(f'\n{len(files)} checkpoint(s) in {ckpt_dir}')

Once `final.pt` exists, move on to `02_eval.ipynb` to evaluate this run.